# SA-CCR on the five official netting sets

- **Level:** 2, The method in practice
- **Prerequisites:** notebook 02 of this topic (collateral and the margin period of risk)
- **Reading time:** about 55 minutes

## Objectives

By the end of this notebook you can:

- state the SA-CCR formula $EAD = 1.4\,(RC + PFE)$ and each piece of it: replacement cost, multiplier, aggregate add-on;
- compute the supervisory duration, the supervisory delta of a linear trade and of an option, and the maturity factor, margined or not;
- compute the interest-rate add-on with its three maturity buckets and offset, the credit add-on with its systematic and idiosyncratic terms, and the commodity add-on;
- reproduce the five examples of the Basel framework (CRE99) from their trades, and compare each intermediate with the printed, rounded value;
- say what the engine does not implement of SA-CCR.

## Setup

Install the pinned environment once, in a virtual environment, with the familiar command:

```bash
pip install -r notebooks/requirements.txt
```

or only the packages this notebook uses, with the same exact pins:

```bash
pip install riskvalidation==1.0.1 numpy==2.4.6
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` on Windows) builds `.venv-notebooks` for you. The next cell imports the packages and prints their versions.

In [1]:
# Import the tools and print the versions they run with, to compare them with notebooks/requirements.txt
import importlib.metadata as md
import json
from pathlib import Path

import numpy as np
import riskvalidation as rv
from riskvalidation.ccr import collateral as col
from riskvalidation.regulatory import sa_ccr

for name in ("riskvalidation", "numpy"):
    print(name, md.version(name))

riskvalidation 1.0.1
numpy 2.4.6


## Data

The data are the regulatory calculators document, `data/derived/regulatory/calculators.json`, schema `contraste.regulatory/v1`. It is not a market dataset: its truth status is `published-answer`. It holds the trades and inputs of the five worked examples of the Basel framework chapter CRE99 (netting sets 1 to 5, transcribed from the BIS page; amounts in USD thousands), the answers the framework prints (rounded), and the answers the engine recomputed when the document was baked, with the paragraph and the in-force date of each rule (2023-01-01).

- **File read:** `data/derived/regulatory/calculators.json`, keys `official_examples`, `parameter_tables` and `provenance`.
- **What is transcribed and what is recomputed:** every input has `basis: transcribed` and every output `basis: recomputed`; the document's `published` list holds what the BIS prints, with its rounding.
- **How we compare:** this notebook recomputes each output from the trades with the engine's SA-CCR functions and compares it with the committed recomputation (exactly) and with the printed value (after the printed rounding).

The next cell loads the document and lists the examples.

In [2]:
# Load the regulatory calculators document (the notebook sits three folders below the repository root) and list its examples
ROOT = Path("../../..").resolve()
doc = json.loads((ROOT / "data" / "derived" / "regulatory" / "calculators.json").read_text(encoding="utf-8"))

print("document:", doc["document_id"], "| engine that baked it:", doc["engine"]["version"], "| installed", rv.__version__)
print("truth status:", doc["provenance"]["truth_status"], "| lane:", doc["lane"]["lane"])
examples = {e["id"]: e for e in doc["official_examples"]}
for ident, e in examples.items():
    print(f"{ident:28s} {e['calculator']:12s} {len(e['trades']):2d} trades | {e['paragraph_range']:22s} | {e['title']['en']}")

document: regulatory-calculators-1.0.1 | engine that baked it: 1.0.1 | installed 1.0.1
truth status: published-answer | lane: precompute
cre99-netting-set-1          sa-ccr        3 trades | CRE52.10 to CRE52.57   | CRE99 netting set 1: interest rate swaps and a swaption, unmargined
cre99-netting-set-2          sa-ccr        3 trades | CRE52.61 to CRE52.65   | CRE99 netting set 2: credit derivatives, unmargined
cre99-netting-set-3          sa-ccr        3 trades | CRE52.70 to CRE52.71   | CRE99 netting set 3: commodity forwards and swaps
cre99-netting-set-4          sa-ccr        6 trades | CRE52.24 to CRE52.25   | CRE99 netting set 4: netting sets 1 and 2 combined
cre99-netting-set-5          sa-ccr        6 trades | CRE52.18 to CRE52.52   | CRE99 netting set 5: margined interest rate and commodity
rbc20-13-output-floor        output-floor  0 trades | RBC20.4 and RBC20.11 to RBC20.13 | RBC20.13 output floor
rbc20-13-output-floor-capped output-floor  0 trades | RBC90.1 to RBC90.2     |

## Method

### Step 1: the formula and the supervisory parameters

SA-CCR (CRE52, in force 2023-01-01) measures the exposure at default of a netting set as

$$EAD = \alpha\,(RC + PFE), \qquad \alpha = 1.4, \qquad PFE = \text{multiplier} \times \text{AddOn}^{aggregate}$$

where RC is the replacement cost (the loss today if the counterparty defaulted, net of collateral), the aggregate add-on is the sum of the add-ons of the five asset classes, and the multiplier lets over-collateralisation reduce the PFE:

$$\text{multiplier} = \min\Big(1,\ \text{Floor} + (1 - \text{Floor})\,e^{(V - C)/(2(1 - \text{Floor})\,\text{AddOn}^{aggregate})}\Big), \qquad \text{Floor} = 5\%$$

The supervisory factor, correlation and option volatility of each asset class are the numbers of the table of CRE52.72. The engine keeps them as data with their paragraph, table and in-force date, in `RegulatoryParameter` objects. The next cell reads some of them, and checks all the rows of the committed table against the engine's.

In [3]:
# Read some supervisory parameters with their paragraph, and check every row of the committed table against the engine
for key, name in (("interest-rate", "supervisory_factor"), ("fx", "supervisory_factor"), ("credit-single-name-AA", "supervisory_factor"),
                  ("credit-single-name-BBB", "supervisory_factor"), ("credit-index-IG", "correlation"),
                  ("commodity-oil-gas-metals-agricultural-other", "supervisory_factor"), ("commodity-oil-gas-metals-agricultural-other", "correlation")):
    p = sa_ccr.supervisory_parameter(key, name)
    print(f"{key:46s} {name:18s} {p.value:7.4f} | {p.paragraph} | {p.table} | {p.in_force}")

table = next(t for t in doc["parameter_tables"] if t["id"] == "cre52-72-table-2")
same = sum(sa_ccr.supervisory_parameter(r["key"], r["parameter"]).value == r["value"] for r in table["rows"])
print("rows of CRE52.72 Table 2 in the document:", len(table["rows"]), "| equal to the engine:", same)
print("alpha:", sa_ccr.ALPHA, "| multiplier floor:", sa_ccr.MULTIPLIER_FLOOR, "| in force:", sa_ccr.CRE52_IN_FORCE)

interest-rate                                  supervisory_factor  0.0050 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
fx                                             supervisory_factor  0.0400 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
credit-single-name-AA                          supervisory_factor  0.0038 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
credit-single-name-BBB                         supervisory_factor  0.0054 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
credit-index-IG                                correlation         0.8000 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
commodity-oil-gas-metals-agricultural-other    supervisory_factor  0.1800 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
commodity-oil-gas-metals-agricultural-other    correlation         0.4000 | CRE52.72 | CRE52.72 Table 2 | 2023-01-01
rows of CRE52.72 Table 2 in the document: 43 | equal to the engine: 43
alpha: 1.4 | multiplier floor: 0.05 | in force: 2023-01-01


### Step 2: duration, delta and maturity factor

Three small functions turn a trade into its effective notional (CRE52.34 to CRE52.52):

- the **supervisory duration** of a trade that starts at $S$ and ends at $E$ (years) is $SD = (e^{-0.05 S} - e^{-0.05 E})/0.05$; it multiplies the notional of interest-rate and credit trades;
- the **supervisory delta** is $\pm 1$ for a linear trade (positive for the payer of fixed, the protection buyer and the long position), and for an option the normal-distribution value $\pm\Phi(\pm d)$ with $d = (\ln(P/K) + \tfrac12 \sigma^2 T)/(\sigma\sqrt{T})$, the sign and the side of $\Phi$ set by buying or selling a call or a put;
- the **maturity factor** is $MF = \sqrt{\min(M, 1)}$ for an unmargined trade and $MF = 1.5\sqrt{MPoR/250}$ for a margined netting set, with the MPoR in business days.

The next cell evaluates them on the trades of the first example and on a few special values.

In [4]:
# Duration, delta and maturity factor on the trades of netting set 1, plus the margined factor and an option delta
ns1 = examples["cre99-netting-set-1"]
terms1 = [{v["name"]: v["value"] for v in tr["terms"]} for tr in ns1["trades"]]
print(f"{'trade':8s}{'start':>7s}{'end':>6s}{'SD':>9s}{'MF':>7s}  bucket")
for tr, t in zip(ns1["trades"], terms1):
    print(f"{tr['trade_id']:8s}{t['start_years']:7.0f}{t['end_years']:6.0f}{sa_ccr.supervisory_duration(t['start_years'], t['end_years']):9.4f}"
          f"{sa_ccr.maturity_factor(t['end_years']):7.3f}  {sa_ccr.maturity_bucket(t['end_years'])}")

print("margined maturity factor, MPoR 14 and 10 days:", round(sa_ccr.maturity_factor(5.0, 14), 6), round(sa_ccr.maturity_factor(5.0, 10), 6))
swaption = terms1[2]
print("swaption terms:", {k: swaption[k] for k in ("option_type", "option_price", "option_strike", "option_volatility", "option_expiry_years")})
delta_put = sa_ccr.supervisory_delta_option(swaption["option_price"], swaption["option_strike"], swaption["option_volatility"],
                                            swaption["option_expiry_years"], False, True)
print("delta of the bought put:", round(delta_put, 6), "| sold put:", round(sa_ccr.supervisory_delta_option(
    swaption["option_price"], swaption["option_strike"], swaption["option_volatility"], swaption["option_expiry_years"], False, False), 6))

trade     start   end       SD     MF  bucket
T1            0    10   7.8694  1.000  B3
T2            0     4   3.6254  1.000  B2
T3            1    11   7.4856  1.000  B3
margined maturity factor, MPoR 14 and 10 days: 0.354965 0.3
swaption terms: {'option_type': 'bought-put', 'option_price': 0.06, 'option_strike': 0.05, 'option_volatility': 0.5, 'option_expiry_years': 1}
delta of the bought put: -0.269395 | sold put: 0.269395


### Step 3: the interest-rate add-on (netting set 1)

Each interest-rate trade has an effective notional $\delta \times N \times SD \times MF$. Trades in one currency form a hedging set. Inside it they fall in three maturity buckets (B1 below one year, B2 from one to five, B3 above five years, by the end date) and the buckets offset each other by the formula of CRE52.57:

$$EN = \sqrt{D_1^2 + D_2^2 + D_3^2 + 1.4\,D_1 D_2 + 1.4\,D_2 D_3 + 0.6\,D_1 D_3}$$

with $D_k$ the sum of the effective notionals in bucket $k$; the add-on of the hedging set is $0.5\% \times EN$. The engine's `ir_addon` returns the trade rows, the hedging-set rows and the total, each with its intermediates. The swaption of netting set 1 is in EUR, so it forms a second hedging set. The next cell builds the trades and prints every intermediate.

In [5]:
# Build the interest-rate trades of netting set 1 with their supervisory deltas and print every intermediate of ir_addon
LINEAR_DELTA = {"pay-fixed-receive-floating": 1.0, "pay-floating-receive-fixed": -1.0, "protection-buyer": 1.0, "protection-seller": -1.0, "long": 1.0, "short": -1.0}
ir_trades = []
for t in terms1:
    delta = delta_put if t.get("option_type", "none") != "none" else LINEAR_DELTA[t["position"]]
    ir_trades.append(sa_ccr.IrTrade(t["currency"], t["start_years"], t["end_years"], t["notional"], delta))
ir = sa_ccr.ir_addon(ir_trades)

print(f"{'set':5s}{'delta':>8s}{'N*SD':>12s}{'MF':>7s}{'bucket':>7s}{'eff. notional':>16s}")
for row in ir.trades:
    print(f"{row.hedging_set:5s}{row.delta:8.4f}{row.adjusted_notional:12.2f}{row.maturity_factor:7.3f}{row.bucket:>7s}{row.effective_notional:16.4f}")
for h in ir.hedging_sets:
    print(f"hedging set {h.hedging_set}: D1 = {h.d_b1:.2f}, D2 = {h.d_b2:.2f}, D3 = {h.d_b3:.2f} | EN = {h.effective_notional:.4f} | factor {h.supervisory_factor} | add-on {h.addon:.4f}")
out1 = {o["name"]: o["value"] for o in ns1["outputs"]}
print("interest-rate add-on:", round(ir.addon, 6), "| committed", round(out1["addon_ir"], 6), "| difference", abs(ir.addon - out1["addon_ir"]))
print("without the offset between buckets:", round(sa_ccr.ir_addon(ir_trades, offset=False).addon, 4))

set     delta        N*SD     MF bucket   eff. notional
USD    1.0000    78693.87  1.000     B3      78693.8681
USD   -1.0000    36253.85  1.000     B2     -36253.8494
EUR   -0.2694    37427.96  1.000     B3     -10082.9138
hedging set USD: D1 = 0.00, D2 = -36253.85, D3 = 78693.87 | EN = 59269.9635 | factor 0.005 | add-on 296.3498
hedging set EUR: D1 = 0.00, D2 = 0.00, D3 = -10082.91 | EN = 10082.9138 | factor 0.005 | add-on 50.4146
interest-rate add-on: 346.764386 | committed 346.764386 | difference 0.0
without the offset between buckets: 625.1532


### Step 4: the credit add-on (netting set 2)

Credit trades have effective notional $\delta \times N \times SD \times MF$ too, but they are aggregated by reference entity and rated: the entity add-on is $A_e = SF_e \times EN_e$ with the supervisory factor of its rating (a single name AA, a single name BBB, an investment-grade index) and a correlation $\rho_e$ with the systematic factor (0.5 for a single name, 0.8 for an index). The add-on of the asset class is (CRE52.61)

$$\text{AddOn}^{credit} = \sqrt{\Big(\sum_e \rho_e A_e\Big)^2 + \sum_e (1 - \rho_e^2) A_e^2}$$

The two terms under the root are the systematic part and the idiosyncratic part; the BIS prints them. The engine's `credit_addon_terms` returns both and the add-on. The next cell computes them for netting set 2 and compares with the printed values.

In [6]:
# Effective notional per credit entity, then the systematic and idiosyncratic terms and the add-on of netting set 2
ns2 = examples["cre99-netting-set-2"]
terms2 = [{v["name"]: v["value"] for v in tr["terms"]} for tr in ns2["trades"]]
entities = {}
for t in terms2:
    en = sa_ccr.credit_trade_effective_notional(LINEAR_DELTA[t["position"]], t["notional"], t["start_years"], t["end_years"])
    row = entities.setdefault(t["entity"], [t["rating"], t["reference_kind"] == "index", 0.0])
    row[2] += en
    print(f"{t['entity']:10s} {t['rating']:4s} {t['position']:16s} effective notional {en:12.4f}")
credit_terms = sa_ccr.credit_addon_terms([sa_ccr.CreditEntity(rating, is_index, en) for rating, is_index, en in entities.values()])
out2 = {o["name"]: o["value"] for o in ns2["outputs"]}
pub2 = {p["name"]: p for p in ns2["published"]}
print("systematic term (sum rho A)^2:", round(credit_terms.systematic, 4), "| printed", pub2["term_1"]["value"])
print("idiosyncratic term sum (1 - rho^2) A^2:", round(credit_terms.idiosyncratic, 4), "| printed", pub2["term_2"]["value"])
print("credit add-on:", round(credit_terms.addon, 6), "| committed", round(out2["addon_credit"], 6), "| printed", pub2["addon_credit"]["value"])

Firm A     AA   protection-buyer effective notional   27858.4047
Firm B     BBB  protection-seller effective notional  -51836.3559
CDX.IG 5y  IG   protection-buyer effective notional   44239.8434
systematic term (sum rho A)^2: 2252.635 | printed 2253
idiosyncratic term sum (1 - rho^2) A^2: 77344.0428 | printed 77344
credit add-on: 282.128832 | committed 282.128832 | printed 282


### Step 5: the commodity add-on (netting set 3)

For commodities the hedging sets are energy, metals, agriculture and so on; inside a set, trades in the same commodity type are added with their sign, and the types of a set aggregate with the same systematic and idiosyncratic formula, using the factor and correlation of the type (0.18 and 0.4 for oil, gas, metals and agricultural; electricity has its own row). The add-on of the class is the sum over hedging sets. The effective notional of a trade is $\delta \times \text{adjusted notional} \times MF$. The engine's `commodity_trade_effective_notional` and `commodity_addon` do it. The next cell computes netting set 3.

In [7]:
# Effective notional per commodity type, then the add-on per hedging set and in total for netting set 3
ns3 = examples["cre99-netting-set-3"]
terms3 = [{v["name"]: v["value"] for v in tr["terms"]} for tr in ns3["trades"]]
sets3 = {}
for t in terms3:
    en = sa_ccr.commodity_trade_effective_notional(LINEAR_DELTA[t["position"]], t["adjusted_notional"], t["maturity_years"])
    key = "commodity-electricity" if t["commodity_type"] == "electricity" else "commodity-oil-gas-metals-agricultural-other"
    sets3.setdefault(t["hedging_set"], {}).setdefault(t["commodity_type"], [key, 0.0])[1] += en
    print(f"{t['hedging_set']:7s}{t['commodity_type']:10s}{t['position']:6s} maturity {t['maturity_years']:5.2f} effective notional {en:12.4f}")
by_set = {hs: [sa_ccr.CommodityEntity(sa_ccr.supervisory_parameter(k, "supervisory_factor").value, sa_ccr.supervisory_parameter(k, "correlation").value, en)
               for k, en in types.values()] for hs, types in sets3.items()}
out3 = {o["name"]: o["value"] for o in ns3["outputs"]}
for hs, entities3 in by_set.items():
    print(f"add-on of the hedging set {hs}:", round(sa_ccr.commodity_addon({hs: entities3}), 4), "| committed", round(out3["addon_" + hs], 4))
commodity_total = sa_ccr.commodity_addon(by_set)
print("commodity add-on:", round(commodity_total, 4), "| committed", round(out3["addon_commodity"], 4))

energy crude-oil long   maturity  0.75 effective notional    8660.2540
energy crude-oil short  maturity  2.00 effective notional  -20000.0000
metals silver    long   maturity  5.00 effective notional   10000.0000
add-on of the hedging set energy: 2041.1543 | committed 2041.1543
add-on of the hedging set metals: 1800.0 | committed 1800.0
commodity add-on: 3841.1543 | committed 3841.1543


### Step 6: replacement cost, multiplier and EAD for the unmargined sets

With the add-ons, the netting set needs its value $V$ (the sum of the market values of its trades) and the collateral $C$. For an unmargined set, $RC = \max(V - C, 0)$. The multiplier is 1 when $V - C \ge 0$ and falls towards the 5% floor as the set is more over-collateralised or more out of the money: in set 2, $V = -20$ gives 0.965. The engine's `replacement_cost`, `multiplier` and `netting_set_ead` apply it. The next cell computes sets 1, 2 and 3, and set 4, which is sets 1 and 2 put together: its add-on is the sum of the interest-rate add-on of set 1 and the credit add-on of set 2, because add-ons add across asset classes, while its value and RC net the two sets.

In [8]:
# RC, multiplier, PFE and EAD of sets 1 to 3, and set 4 as sets 1 and 2 together, against the committed recomputation and the printed value
v = {1: sum(t["market_value"] for t in terms1), 2: sum(t["market_value"] for t in terms2), 3: sum(t["market_value"] for t in terms3)}
v[4] = v[1] + v[2]
addons = {1: sa_ccr.AddOnByClass(interest_rate=ir.addon), 2: sa_ccr.AddOnByClass(credit=credit_terms.addon),
          3: sa_ccr.AddOnByClass(commodity=commodity_total)}
addons[4] = sa_ccr.AddOnByClass(interest_rate=ir.addon, credit=credit_terms.addon)
result = {n: sa_ccr.netting_set_ead(v[n], 0.0, addons[n]) for n in addons}

print(f"{'':6s}{'V':>6s}{'RC':>8s}{'AddOn':>10s}{'mult.':>8s}{'PFE':>10s}{'EAD':>10s}{'printed EAD':>14s}{'committed - ours':>20s}")
for n in addons:
    e = examples[f"cre99-netting-set-{n}"]
    committed = {o["name"]: o["value"] for o in e["outputs"]}
    printed = {p["name"]: p["value"] for p in e["published"]}
    r_ = result[n]
    print(f"set {n}{v[n]:6.0f}{r_.rc:8.2f}{r_.addon_aggregate:10.4f}{r_.multiplier:8.4f}{r_.pfe:10.4f}{r_.ead:10.4f}{printed['ead']:14.0f}{committed['ead'] - r_.ead:20.2e}")
print("multiplier of set 2 from the formula inputs:", round(sa_ccr.multiplier(-20.0, credit_terms.addon), 6))
print("set 4: RC nets the two sets (V = 40) while the add-ons add:", result[4].rc, round(result[4].addon_aggregate, 4))

           V      RC     AddOn   mult.       PFE       EAD   printed EAD    committed - ours
set 1    60   60.00  346.7644  1.0000  346.7644  569.4701           569            0.00e+00
set 2   -20    0.00  282.1288  0.9652  272.3131  381.2383           381            0.00e+00
set 3    20   20.00 3841.1543  1.0000 3841.1543 5405.6160          5406            0.00e+00
set 4    40   40.00  628.8932  1.0000  628.8932  936.4505           936            0.00e+00
multiplier of set 2 from the formula inputs: 0.965208
set 4: RC nets the two sets (V = 40) while the add-ons add: 40.0 628.8932


### Step 7: a margined netting set (netting set 5)

Set 5 joins the interest-rate trades of set 1 with the commodity trades of set 3 under a margin agreement: $V = 80$, collateral $C = 200$, threshold $TH = 0$, minimum transfer amount $MTA = 5$, net independent collateral amount $NICA = 150$, daily margining replaced by margining every $N=5$ business days. Three things change:

- the MPoR is $F + N - 1$ with the floor $F = 10$ (CRE52.50 and CRE53.26, notebook 02): $10 + 5 - 1 = 14$ business days;
- every maturity factor becomes $1.5\sqrt{MPoR/250}$, the same for all trades;
- the replacement cost is $RC = \max(V - C,\ TH + MTA - NICA,\ 0)$, so a netting set cannot have a replacement cost below the margin terms.

The next cell computes all of it.

In [9]:
# Netting set 5: the MPoR from the re-margining interval, the margined add-ons, RC, multiplier and EAD
ns5 = examples["cre99-netting-set-5"]
ins5 = {i["name"]: i["value"] for i in ns5["inputs"]}
mpor = col.mpor_with_remargining(10, ins5["margin_frequency_business_days"])
print("MPoR = 10 + 5 - 1 =", mpor, "| input of the document:", ins5["mpor_business_days"], "| maturity factor:", round(sa_ccr.maturity_factor(1.0, mpor), 6))

terms5 = [{v_["name"]: v_["value"] for v_ in tr["terms"]} for tr in ns5["trades"]]
ir5, com5 = [], {}
for t in terms5:
    if t["asset_class"] == "interest-rate":
        delta = delta_put if t.get("option_type", "none") != "none" else LINEAR_DELTA[t["position"]]
        ir5.append(sa_ccr.IrTrade(t["currency"], t["start_years"], t["end_years"], t["notional"], delta))
    else:
        en = sa_ccr.commodity_trade_effective_notional(LINEAR_DELTA[t["position"]], t["adjusted_notional"], t["maturity_years"], mpor)
        com5.setdefault(t["hedging_set"], {}).setdefault(t["commodity_type"], [("commodity-oil-gas-metals-agricultural-other"), 0.0])[1] += en
addon_ir5 = sa_ccr.ir_addon(ir5, mpor).addon
addon_com5 = sa_ccr.commodity_addon({hs: [sa_ccr.CommodityEntity(sa_ccr.supervisory_parameter(k, "supervisory_factor").value,
                                                                  sa_ccr.supervisory_parameter(k, "correlation").value, en) for k, en in types.values()]
                                     for hs, types in com5.items()})
res5 = sa_ccr.netting_set_ead(ins5["v"], ins5["c"], sa_ccr.AddOnByClass(interest_rate=addon_ir5, commodity=addon_com5), True, ins5["th"], ins5["mta"], ins5["nica"])
out5 = {o["name"]: o["value"] for o in ns5["outputs"]}
pub5 = {p["name"]: p["value"] for p in ns5["published"]}
print("margin terms TH + MTA - NICA:", ins5["th"] + ins5["mta"] - ins5["nica"], "| V - C =", ins5["v"] - ins5["c"], "| RC =", res5.rc)
print("add-ons: interest rate, commodity, aggregate:", round(addon_ir5, 4), round(addon_com5, 4), round(res5.addon_aggregate, 4))
print("multiplier:", round(res5.multiplier, 6), "| PFE:", round(res5.pfe, 4), "| EAD:", round(res5.ead, 4), "| printed EAD:", pub5["ead"], "| committed - ours:", out5["ead"] - res5.ead)

MPoR = 10 + 5 - 1 = 14 | input of the document: 14 | maturity factor: 0.354965
margin terms TH + MTA - NICA: -145 | V - C = -120 | RC = 0.0
add-ons: interest rate, commodity, aggregate: 123.0891 1277.8732 1400.9624
multiplier: 0.958123 | PFE: 1342.2947 | EAD: 1879.2126 | printed EAD: 1879 | committed - ours: 0.0


### Step 8: every intermediate against the printed value

The framework prints each intermediate rounded (to the nearest integer, or to three or four decimals). The test of a reproduction is that our unrounded number, rounded as the framework rounds it, is the printed one, and that our number equals the committed recomputation. The next cell does both for the five examples, for every output that appears in the `published` list, taking our value from the steps above.

In [10]:
# Compare each reproduced output with the committed recomputation and with the printed value after the printed rounding
ours = {
    "cre99-netting-set-1": {"rc": result[1].rc, "addon_ir": ir.addon, "addon_aggregate": result[1].addon_aggregate, "multiplier": result[1].multiplier,
                            "ead": result[1].ead, "swaption_delta": delta_put, "en_usd": ir.hedging_sets[0].effective_notional, "en_eur": ir.hedging_sets[1].effective_notional},
    "cre99-netting-set-2": {"rc": result[2].rc, "term_1": credit_terms.systematic, "term_2": credit_terms.idiosyncratic, "addon_credit": credit_terms.addon,
                            "addon_aggregate": result[2].addon_aggregate, "multiplier": result[2].multiplier, "ead": result[2].ead},
    "cre99-netting-set-3": {"rc": result[3].rc, "addon_commodity": commodity_total, "addon_aggregate": result[3].addon_aggregate,
                            "multiplier": result[3].multiplier, "ead": result[3].ead,
                            "addon_energy": sa_ccr.commodity_addon({"energy": by_set["energy"]}), "addon_metals": sa_ccr.commodity_addon({"metals": by_set["metals"]})},
    "cre99-netting-set-4": {"rc": result[4].rc, "addon_ir": ir.addon, "addon_credit": credit_terms.addon, "addon_aggregate": result[4].addon_aggregate,
                            "multiplier": result[4].multiplier, "ead": result[4].ead},
    "cre99-netting-set-5": {"rc": res5.rc, "addon_ir": addon_ir5, "addon_commodity": addon_com5, "addon_aggregate": res5.addon_aggregate,
                            "multiplier": res5.multiplier, "ead": res5.ead, "maturity_factor": sa_ccr.maturity_factor(1.0, mpor)},
}
checked = matched = exact = 0
worst = 0.0
for ident, values_ in ours.items():
    committed = {o["name"]: o["value"] for o in examples[ident]["outputs"]}
    for p in examples[ident]["published"]:
        if p["name"] not in values_:
            continue
        digits = {"nearest integer": 0, "3 decimal places": 3, "4 decimal places": 4}.get(p["rounding"])
        printed_ok = values_[p["name"]] == p["value"] if digits is None else round(values_[p["name"]], digits) == p["value"]
        gap = abs(values_[p["name"]] - committed[p["name"]]) / max(abs(committed[p["name"]]), 1e-300)
        worst = max(worst, gap)
        checked += 1
        matched += int(printed_ok)
        exact += int(gap < 1e-9)
print("printed values checked:", checked, "| equal after the printed rounding:", matched, "| equal to the committed recomputation (1e-9):", exact)
print("largest relative gap to the committed recomputation:", worst)
print("EAD of the five sets, ours:", {k[-1]: round(val["ead"], 1) for k, val in ours.items()}, "| printed:", {k[-1]: {p["name"]: p["value"] for p in examples[k]["published"]}["ead"] for k in ours})

printed values checked: 34 | equal after the printed rounding: 34 | equal to the committed recomputation (1e-9): 34
largest relative gap to the committed recomputation: 0.0
EAD of the five sets, ours: {'1': 569.5, '2': 381.2, '3': 5405.6, '4': 936.5, '5': 1879.2} | printed: {'1': 569, '2': 381, '3': 5406, '4': 936, '5': 1879}


## Results

- **The parameters.** All 43 rows of CRE52.72 Table 2 in the committed document equal the engine's, and each carries its paragraph, table and in-force date (2023-01-01), so a number on a screen can show where it comes from.
- **Interest rates (set 1).** The 10-year payer has duration 7.869 and effective notional +78,694; the 4-year receiver has duration 3.625 and falls in bucket B2 with -36,254; the EUR swaption has delta -0.2694, a bought put. The USD hedging set offsets the two buckets to an effective notional of 59,270 and an add-on of 296.35; with the EUR set the interest-rate add-on is 346.76. Without the offset between buckets it would be 625.15: the offset formula removes 44.5%.
- **Credit (set 2).** The systematic term is 2,252.6 and the idiosyncratic one 77,344.0, equal to the printed 2,253 and 77,344; the add-on is 282.13. The multiplier is 0.9652 because $V=-20$ is below zero.
- **Commodities (set 3).** The energy set gives 2,041.15 and the metals set 1,800.00; the class add-on is 3,841.15.
- **Aggregation (set 4).** Set 4 is sets 1 and 2 together. Its aggregate add-on, 628.89, is the sum 346.76 + 282.13, because add-ons add across asset classes, while its replacement cost, 40, nets the values 60 and -20. The EAD is 936.45.
- **Margin (set 5).** The MPoR is $10 + 5 - 1 = 14$ business days, the document's input, and every maturity factor becomes 0.3550. The replacement cost is $\max(-120, -145, 0) = 0$. The multiplier is 0.9581 and the EAD 1,879.21.
- **The five EAD** are 569.47, 381.24, 5,405.62, 936.45 and 1,879.21 (USD thousands), which round to the printed 569, 381, 5,406, 936 and 1,879. Of the 34 printed intermediates, all 34 agree after the printed rounding and all 34 equal the committed recomputation to the last digit.

What the engine does not implement of SA-CCR is stated in its documentation: the shift of strike and price for negative rates (CRE52.40), the vega of FX and commodity options, basis and volatility transactions, and the sign rules for FX trades beyond a signed effective notional the caller supplies. The five examples use only interest-rate, credit and commodity trades, so they do not exercise the FX and equity add-ons, which the engine offers as `fx_addon` and `equity_addon`.

## Summary

- $EAD = 1.4\,(RC + \text{multiplier} \times \text{AddOn})$; the add-on is the sum over five asset classes, each built from effective notionals $\delta \times N \times SD \times MF$.
- Interest-rate trades offset across three maturity buckets, credit and commodity trades across entities by a systematic plus idiosyncratic formula, and add-ons add across classes.
- A margin agreement changes three things: the MPoR $F + N - 1$, the maturity factor $1.5\sqrt{MPoR/250}$ and the replacement cost $\max(V - C, TH + MTA - NICA, 0)$.
- The five CRE99 examples are reproduced from their trades: 34 of 34 printed intermediates agree and the numbers equal the committed recomputation exactly.

**What to try next.** Remove the offset (`offset=False`) in set 5 and see how much the interest-rate add-on grows; change the swaption to a sold put and watch the EUR hedging set flip sign. Then continue with notebook 05.

## References

- BCBS, Basel Framework, CRE52 Standardised approach to counterparty credit risk (in force 2023-01-01, published 2020-06-05). https://www.bis.org/committees/bcbs/basel-framework/standard/cre/52/inforce/2023-01-01/published/2020-06-05
- BCBS, Basel Framework, CRE99 Illustrative examples of the standardised approach to counterparty credit risk (in force 2023-01-01). https://www.bis.org/committees/bcbs/basel-framework/standard/cre/99/inforce/2023-01-01/published/2020-03-27
- BCBS, Basel Framework, CRE53 Internal model method for counterparty credit risk (in force 2023-01-01, published 2020-06-05). https://www.bis.org/basel_framework/chapter/CRE/53.htm?inforce=20230101&published=20200605